# Gruppen-CV für den hierarchischen Ansatz

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** #170 für die hierarchische SVM wiederholen (#193). Wie stark bricht sie bei unbekannten Aufnahmen, Monaten und Regionen ein, und welche Stufe (Kultur oder Stadium) leidet mehr?

**Vorgehen:** Alle Bänder und Sentinel-2, je vier Fold-Arten, CV auf dem 80/20-Split, gleiches Raster wie in `08_datenreduktion/03_jd_hierarchisch-reduktion`. Zum Vergleich werden die kombinierten Werte aus #170 aus MLflow gelesen (Experiment `split-leakage`).

In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Any

import matplotlib.pyplot as plt
import mlflow
import pandas as pd

from awp2.config import (
    MLFLOW_SPLIT_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    MODEL_DOCS_FIGURES_DIR,
    REDUCTION_VARIANT_COL,
)
from awp2.evaluation import METRIC_LABELS
from awp2.experiment import TuneConfig
from awp2.models import make_hierarchical_svm
from awp2.plots import save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import CURVE_METRICS, reduction_curve, reduction_variants

GROUPINGS = ("random", "aez_month", "month", "aez")
base = PreprocessingConfig(use_spectral_standard_scale=True)
grid: dict[str, list[Any]] = {
    "crop_estimator__C": [30.0],
    "crop_estimator__gamma": [0.01, "scale"],
    "stage_estimator__C": [100.0],
    "stage_estimator__gamma": [0.003, "scale"],
}
standard = {v.name: v for v in reduction_variants(base)}
variants = [standard["all_bands"], standard["sentinel2"]]

## 1. Fold-Arten × Eingaben

Jede Kombination wird neu getunt und je Fold für Kultur, Stadium und kombiniert bewertet.

In [ ]:
curves = []
for grouping in GROUPINGS:
    curve = reduction_curve(
        make_hierarchical_svm(),
        variants,
        TuneConfig(
            name=f"hier_split_{grouping}",
            approach="hierarchical",
            experiment=MLFLOW_SPLIT_EXPERIMENT,
            description=f"Split-Leakage-Check hierarchisch (#193): Folds {grouping}",
            param_grid=grid,
            preprocessing=base,
            folds=grouping,
        ),
    )
    curves.append(curve.assign(folds=grouping))
results = pd.concat(curves, ignore_index=True)
results.pivot(index="folds", columns=REDUCTION_VARIANT_COL, values=list(CURVE_METRICS)).loc[
    list(GROUPINGS)
].round(3)

## 2. Vergleich mit dem kombinierten Ansatz (#170)

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
combined_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_SPLIT_EXPERIMENT],
    filter_string=(
        "params.`split.validation_size` = '0.2' and tags.`run.kind` = 'tuning' "
        "and attributes.run_name LIKE 'split_%'"
    ),
)
combined = {
    name.removeprefix("split_"): score
    for name, score in zip(
        combined_runs["tags.mlflow.runName"],
        combined_runs["metrics.cv_bacc_combined_mean"],
        strict=True,
    )
}
comparison = pd.DataFrame(
    {
        "hierarchisch": {
            (row["folds"], row[REDUCTION_VARIANT_COL]): row["bacc_combined"]
            for _, row in results.iterrows()
        }
    }
)
comparison["kombiniert"] = [
    combined.get(f"{grouping}_{variant}") for grouping, variant in comparison.index
]
comparison["Differenz"] = comparison["hierarchisch"] - comparison["kombiniert"]
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, metric in zip(axes, CURVE_METRICS, strict=True):
    means = results.pivot(index="folds", columns=REDUCTION_VARIANT_COL, values=metric)
    stds = results.pivot(index="folds", columns=REDUCTION_VARIANT_COL, values=f"{metric}_std")
    means.loc[list(GROUPINGS)].plot.bar(
        ax=ax, yerr=stds.loc[list(GROUPINGS)], rot=0, capsize=2, width=0.8
    )
    ax.set_title(f"{METRIC_LABELS[metric]} (CV)")
    ax.set_xlabel("Fold-Art")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize="small", loc="lower left")
fig.suptitle("Hierarchisch: zufällige vs. Gruppen-Folds")
fig.tight_layout()
save_doc_figure(fig, "hierarchisch_gruppen_cv", MODEL_DOCS_FIGURES_DIR)

## Fazit

| Fold-Art | Hierarchisch, alle Bänder | Hierarchisch, Sentinel-2 | Kombiniert, alle Bänder (#170) |
| --- | --- | --- | --- |
| zufällig | 0,846 | 0,834 | **0,859** |
| neue Aufnahme (AEZ × Monat) | **0,420** | **0,426** | 0,378 |
| neuer Monat | 0,417 | 0,424 | 0,421 |
| neue Region (AEZ) | 0,269 | 0,294 | 0,284 |

*CV-BAcc kombiniert, 80/20-Split, Experiment `split-leakage`.*

- **Auch hierarchisch bricht die Qualität bei unbekannten Aufnahmen ein** (0,85 → 0,27–0,43). Die Grenze aus #170 gilt für beide Ansätze.
- **Bei neuen Aufnahmen ist der hierarchische Ansatz robuster:** +0,04 gegenüber dem kombinierten (0,420 gegenüber 0,378), obwohl er bei bekannten Zellen 0,012 schlechter ist. Bei neuem Monat und neuer Region liegen beide im Rauschen (Streuung bis ± 0,12).
- **Der Vorteil kommt aus der Kultur-Stufe:** Bei neuen Aufnahmen erkennt das eigene Kultur-Modell 0,72 (kombiniert etwa 0,62). Als eigenes Ziel lernt die Kultur offenbar allgemeinere Merkmale, als wenn sie in 23 kombinierten Klassen mitgelernt wird.
- **Beim Stadium ist es umgekehrt:** Ohne den Monat im Training fällt das Stadium hierarchisch stärker (0,464 gegenüber 0,51 kombiniert). Jedes Stage-Modell sieht nur seine Kultur, also noch weniger Beispiele pro fehlendem Monat.
- **Sentinel-2 ist bei unbekannten Aufnahmen gleichauf oder etwas besser** (0,426 / 0,424 / 0,294 gegenüber 0,420 / 0,417 / 0,269), vor allem beim Stadium. Breite Bänder sind vermutlich weniger empfindlich für die Eigenheiten einzelner Aufnahmen.
- **Für die Modellkarte:** Für die Challenge (Testdaten aus bekannten Zellen) bleibt der kombinierte Ansatz die bessere Wahl. Für neue Aufnahmen wäre der hierarchische Ansatz vorzuziehen, am besten mit Sentinel-2-Eingabe.